# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Create a Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata (as object, not a dictionary)
metadata = dataset.metadata

# Display basic metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and their fields, using their `@id`s. All entity references should be through `@id`.

In [ ]:
# List all record sets available in the dataset by their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset. Please make sure the Croissant schema includes record sets.")
else:
    print("Available record sets (by @id):")
    for rs in record_sets:
        print(f"- {rs['@id']}")

    # For demonstration, display the fields and columns of each record set by their @id
    for rs in record_sets:
        print(f"\nRecord Set @id: {rs['@id']}")
        if 'field' in rs and rs['field']:
            print("  Fields:")
            for f in rs['field']:
                field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
                print(f"    - {field_id}")
        if 'column' in rs and rs['column']:
            print("  Columns:")
            for c in rs['column']:
                column_id = c['@id'] if isinstance(c, dict) and '@id' in c else c
                print(f"    - {column_id}")

## 3. Data Extraction
For each record set, use its `@id` to load records as a pandas DataFrame. All data references are strictly via `@id` fields.

In [ ]:
# Build a list of record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows. Columns (@id): {list(df.columns)}")
        display(df.head())
    else:
        print("  No records found for this record set.")

if dataframes:
    # For demonstration: Pick first available record set for further EDA
    main_record_set_id = next(iter(dataframes))
    print(f"\nProceeding with main record set: {main_record_set_id}")
    print("Available columns (@id):", dataframes[main_record_set_id].columns.tolist())
else:
    print("No records loaded. Cannot proceed to EDA.")

## 4. Exploratory Data Analysis (EDA)
You can filter, transform, and group data using the column `@id`s.
We'll demonstrate common steps such as numeric filtering, normalization, and grouping, referencing data by their correct `@id`s.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if dataframes:
    df = dataframes[main_record_set_id]
    # Attempt to infer numeric columns by checking dtype
    numeric_column_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_column_ids:
        print("No numeric columns detected for filtering or normalization.")
    else:
        # Pick the first numeric column (or specify by known @id)
        numeric_field_id = numeric_column_ids[0]
        print(f"Analyzing numeric field: {numeric_field_id}")

        # Remove NaN values and filter
        threshold = df[numeric_field_id].dropna().mean()  # Use mean as dynamic threshold example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records in '{numeric_field_id}' above threshold ({threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric/grouping-appropriate field (pick the first non-numeric column as example)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped.head())
        else:
            print("No non-numeric fields found for grouping.")
else:
    print("No dataframe loaded for EDA.")

## 5. Visualization
Use `matplotlib` or `seaborn` for data visualization. Here, we'll plot a histogram and a group comparison if possible, using column `@id`s only.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[main_record_set_id]
    if numeric_column_ids:
        numeric_field_id = numeric_column_ids[0]
        # 1. Distribution Histogram
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        if group_field_id:
            # 2. Boxplot by group
            plt.figure(figsize=(8, 4))
            sns.boxplot(y=df[numeric_field_id], x=df[group_field_id].astype(str))
            plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and preprocess a dataset described by a Croissant schema using the `mlcroissant` library.

**Key steps and findings:**
- Metadata and data are loaded with strict referencing via `@id` fields.
- Record sets, fields, and columns have been identified and referenced using their unique IDs.
- Basic EDA and visualizations can be executed fully by referencing entities using their `@id`s to ensure transparency and reproducibility.

You can extend this analysis with more domain-specific processing and reporting as needed.